In [28]:
# Import urllib library to download files from internet
import urllib.request  

# URL of the dataset (FTP server containing web logs)
url = "ftp://ita.ee.lbl.gov/traces/calgary_access_log.gz"

# Download the file from the URL and save it locally
# "calgary_access_log.gz" → name of the file after download
urllib.request.urlretrieve(url, "calgary_access_log.gz")

('calgary_access_log.gz', <email.message.Message at 0x1d53b700640>)

In [29]:
# Import os module to interact with operating system
import os  

# List all files and folders in current working directory
print(os.listdir())

['.astropy', '.conda', '.condarc', '.config', '.continuum', '.docker', '.gitconfig', '.ipynb_checkpoints', '.ipython', '.jupyter', '.matplotlib', '.nbi', '.nuget', '.openjfx', '.redhat', '.vscode', '.wdm', '05092025.ipynb', '3D Objects', 'anaconda3', 'analysis-Copy1.ipynb', 'analysis.ipynb', 'AppData', 'Application Data', 'Assigment2.ipynb', 'Assignment .ipynb', 'Assignment1.ipynb', 'Bike_data .ipynb', 'calgary_access_log.gz', 'Car_data.ipynb', 'Contacts', 'Cookies', 'DataCleaning.ipynb', 'datacleaning22092025.ipynb', 'Dataset With OOPS  .ipynb', 'Documents', 'Downloads', 'durga cls.ipynb', 'ESR', 'Favorites', 'forandwhileloop .ipynb', 'IBA_IOAPDATA', 'input and output.ipynb', 'jobs_links.xlsx', 'Links', 'list.ipynb', 'Local Settings', 'mahi', 'MAP,FILTER,BASIC OPERATION ON DATASET.ipynb', 'Merge.ipynb', 'MicrosoftEdgeBackups', 'microsoft_and_google-Copy1.ipynb', 'microsoft_and_google.ipynb', 'microsoft_automation.ipynb', 'Music', 'My Documents', 'NetHood', 'NTUSER.DAT', 'ntuser.dat.LO

In [30]:
# Import gzip module to work with compressed (.gz) files
import gzip

In [31]:
# Open compressed file in text mode ('rt' = read text)
with gzip.open("calgary_access_log.gz", 'rt', encoding='utf-8', errors='ignore') as f:
    
    # Loop through first 5 lines
    for i in range(5):
        print(f.readline())  # Read one line at a time

local - - [24/Oct/1994:13:41:41 -0600] "GET index.html HTTP/1.0" 200 150

local - - [24/Oct/1994:13:41:41 -0600] "GET 1.gif HTTP/1.0" 200 1210

local - - [24/Oct/1994:13:43:13 -0600] "GET index.html HTTP/1.0" 200 3185

local - - [24/Oct/1994:13:43:14 -0600] "GET 2.gif HTTP/1.0" 200 2555

local - - [24/Oct/1994:13:43:15 -0600] "GET 3.gif HTTP/1.0" 200 36403



In [32]:


# Open file using latin1 encoding (handles special characters better)
with gzip.open("calgary_access_log.gz", 'rt', encoding='latin1') as f:
    
    # Read all lines into a list
    lines = f.readlines()

# Print total number of rows
print("Total rows:", len(lines))

# Print first 5 rows for verification
print(lines[:5])

Total rows: 726739
['local - - [24/Oct/1994:13:41:41 -0600] "GET index.html HTTP/1.0" 200 150\n', 'local - - [24/Oct/1994:13:41:41 -0600] "GET 1.gif HTTP/1.0" 200 1210\n', 'local - - [24/Oct/1994:13:43:13 -0600] "GET index.html HTTP/1.0" 200 3185\n', 'local - - [24/Oct/1994:13:43:14 -0600] "GET 2.gif HTTP/1.0" 200 2555\n', 'local - - [24/Oct/1994:13:43:15 -0600] "GET 3.gif HTTP/1.0" 200 36403\n']


In [33]:
import pandas as pd

pd.set_option('display.max_colwidth', None)   # show full text
pd.set_option('display.max_columns', None)    # show all columns
pd.set_option('display.width', None)    

In [34]:
# Import pandas library
# Pandas is used for handling structured data in table format (DataFrame)
import pandas as pd

# Create a DataFrame from the 'lines' list
# 'lines' contains raw log data where each element is one log entry (string)
# We are creating a single column DataFrame named 'raw_log'
df = pd.DataFrame(lines, columns=["raw_log"])

# Display the first 5 rows of the DataFrame
# This helps us preview the data and verify it loaded correctly
print(df.head())

                                                                       raw_log
0   local - - [24/Oct/1994:13:41:41 -0600] "GET index.html HTTP/1.0" 200 150\n
1       local - - [24/Oct/1994:13:41:41 -0600] "GET 1.gif HTTP/1.0" 200 1210\n
2  local - - [24/Oct/1994:13:43:13 -0600] "GET index.html HTTP/1.0" 200 3185\n
3       local - - [24/Oct/1994:13:43:14 -0600] "GET 2.gif HTTP/1.0" 200 2555\n
4      local - - [24/Oct/1994:13:43:15 -0600] "GET 3.gif HTTP/1.0" 200 36403\n


In [35]:
# Remove leading and trailing spaces from each log entry
df['raw_log'] = df['raw_log'].str.strip()

# Extract structured fields (host, timestamp, filename, http_code, bytes) from raw log using regex
df[['host', 'timestamp', 'filename', 'http_code', 'bytes']] = df['raw_log'].str.extract(
    r'(\S+) \S+ \S+ \[([^\]]+)\] "\S+ (\S+) \S+" (\d{3}) (\S+)'
)

# Rename column 'request' to 'filename' (safety step, even if not present)
df.rename(columns={'request': 'filename'}, inplace=True)

# Convert http_code column to numeric (invalid values become NaN)
df['http_code'] = pd.to_numeric(df['http_code'], errors='coerce')

# Convert bytes column to numeric (handles '-' or invalid values as NaN)
df['bytes'] = pd.to_numeric(df['bytes'], errors='coerce')

# Set pandas display option to show full column content without truncation
pd.set_option('display.max_colwidth', None)

# Remove any extra spaces from timestamp column
df['timestamp'] = df['timestamp'].str.strip()

# Convert timestamp string into datetime format for time-based analysis
df['timestamp'] = pd.to_datetime(
    df['timestamp'],
    format='%d/%b/%Y:%H:%M:%S %z',
    errors='coerce'
)

In [36]:
# Convert timestamp column to datetime format and standardize it to UTC timezone
df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)

# Convert bytes column to numeric values (invalid entries become NaN)
df['bytes'] = pd.to_numeric(df['bytes'], errors='coerce')

# Convert http_code to numeric and store as nullable integer type (Int64 supports NaN)
df['http_code'] = pd.to_numeric(df['http_code'], errors='coerce').astype('Int64')



In [37]:
new_df = df[['host', 'timestamp', 'filename', 'http_code', 'bytes']]

In [38]:
(new_df.isnull().sum() / len(new_df)) * 100

host         0.613425
timestamp    0.613425
filename     0.613425
http_code    0.613425
bytes        8.582999
dtype: float64

In [39]:
new_df = new_df.dropna(subset=['host', 'timestamp', 'filename', 'http_code'])

In [40]:
new_df['bytes'] = new_df['bytes'].fillna(0)

In [41]:
new_df['file_extension'] = new_df['filename'].str.extract(r'\.([a-zA-Z0-9]+)$')

In [42]:
final_df = new_df[['host', 'timestamp', 'file_extension', 'http_code', 'bytes','filename']]

In [43]:
#Analysis questions

In [44]:
# 1  Count of total log records

In [45]:
# Calculate the total number of records (rows) in the DataFrame
total_records = len(final_df)

# Print the total record count to verify dataset size
print(total_records)

722281


In [46]:
# 2 Count of unique hosts

In [47]:
# Calculate the number of unique hosts (distinct users/IPs) in the dataset
unique_hosts = final_df['host'].nunique()

# Print the count of unique hosts to understand user diversity
print(unique_hosts)

3


In [48]:
# 3 Date-wise unique filename counts

In [49]:
# Create a new column 'date' by formatting timestamp into 'day-month-year' string
final_df['date'] = final_df['timestamp'].dt.strftime('%d-%b-%Y')

# Group data by date and count unique filenames accessed per day, then convert result to dictionary
result_dict = (
    final_df.groupby('date')['filename']
    .nunique()
    .to_dict()
)

# Print the dictionary showing date-wise unique file access count
print(result_dict)

C:\Users\VISHNU\AppData\Local\Temp\ipykernel_16656\4152003580.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['date'] = final_df['timestamp'].dt.strftime('%d-%b-%Y')


{'01-Apr-1995': 407, '01-Aug-1995': 663, '01-Dec-1994': 244, '01-Feb-1995': 571, '01-Jan-1995': 82, '01-Jul-1995': 405, '01-Jun-1995': 569, '01-Mar-1995': 519, '01-May-1995': 432, '01-Nov-1994': 436, '01-Oct-1995': 583, '01-Sep-1995': 445, '02-Apr-1995': 394, '02-Aug-1995': 785, '02-Dec-1994': 356, '02-Feb-1995': 619, '02-Jan-1995': 128, '02-Jul-1995': 362, '02-Jun-1995': 546, '02-Mar-1995': 676, '02-May-1995': 694, '02-Nov-1994': 422, '02-Oct-1995': 839, '02-Sep-1995': 307, '03-Apr-1995': 813, '03-Aug-1995': 659, '03-Dec-1994': 152, '03-Feb-1995': 566, '03-Jan-1995': 257, '03-Jul-1995': 460, '03-Jun-1995': 397, '03-Mar-1995': 502, '03-May-1995': 565, '03-Nov-1994': 450, '03-Oct-1995': 833, '03-Sep-1995': 207, '04-Apr-1995': 862, '04-Aug-1995': 739, '04-Dec-1994': 211, '04-Feb-1995': 450, '04-Jan-1995': 313, '04-Jul-1995': 491, '04-Jun-1995': 323, '04-Mar-1995': 364, '04-May-1995': 700, '04-Nov-1994': 417, '04-Oct-1995': 909, '04-Sep-1995': 309, '05-Apr-1995': 830, '05-Aug-1995': 497, 

In [50]:
#4 Number of 404 response codes

In [51]:
# Count the number of records where HTTP status code is 404 (Not Found errors)
count_404 = (final_df['http_code'] == 404).sum()

# Print the total number of 404 errors in the dataset
print(count_404)

23440


In [52]:
#5 Top 15 filenames with 404 responses

In [53]:
# Filter records with 404 errors, count frequency of each filename, and get top 15 most frequent ones
result = (
    final_df[final_df['http_code'] == 404]['filename']
    .value_counts()
    .head(15)
    .items()
)

# Convert the result into a list of tuples (filename, count)
result = list(result)

# Print the top 15 filenames causing 404 errors
print(result)

[('index.html', 4692), ('4115.html', 900), ('1611.html', 649), ('5698.xbm', 585), ('710.txt', 408), ('2002.html', 258), ('2177.gif', 193), ('10695.ps', 161), ('6555.html', 153), ('487.gif', 152), ('151.html', 149), ('3414.gif', 148), ('488.gif', 148), ('40.html', 148), ('9678.gif', 142)]


In [54]:
# 6 Top 15 file extensions with 404 responses

In [55]:
# Filter only 404 errors
df_404 = final_df[final_df['http_code'] == 404]

# Group by file_extension and count occurrences
result = (
    df_404
    .groupby('file_extension')
    .size()
    .sort_values(ascending=False)
    .head(15)
)

# Convert to required format: list of tuples
top_15_extensions = list(result.items())

print(top_15_extensions)

[('html', 12137), ('gif', 7202), ('xbm', 824), ('ps', 754), ('jpg', 520), ('txt', 496), ('GIF', 135), ('htm', 107), ('cgi', 77), ('com', 45), ('Z', 41), ('dvi', 40), ('ca', 36), ('hmtl', 30), ('util', 29)]


In [56]:
# 7 Total bandwidth transferred per day for July 1995

In [57]:
# Convert timestamp column to datetime format (invalid values become NaT)
final_df['timestamp'] = pd.to_datetime(final_df['timestamp'], errors='coerce')

# Convert bytes column to numeric (invalid values become NaN)
final_df['bytes'] = pd.to_numeric(final_df['bytes'], errors='coerce')

# Filter records only for July 1995 using year and month from timestamp
df_july = final_df[
    (final_df['timestamp'].dt.year == 1995) &
    (final_df['timestamp'].dt.month == 7)
]

# Create a new column 'day' by formatting timestamp into readable date (day-month-year)
df_july['day'] = df_july['timestamp'].dt.strftime('%d-%b-%Y')

# Group by day, sum total bytes transferred per day, convert to int, and store as dictionary
result_dict = df_july.groupby('day')['bytes'].sum().astype(int).to_dict()

# Print the dictionary showing daily total bytes for July 1995
print(result_dict)

C:\Users\VISHNU\AppData\Local\Temp\ipykernel_16656\1459748089.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['timestamp'] = pd.to_datetime(final_df['timestamp'], errors='coerce')
C:\Users\VISHNU\AppData\Local\Temp\ipykernel_16656\1459748089.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['bytes'] = pd.to_numeric(final_df['bytes'], errors='coerce')


{'01-Jul-1995': 16986893, '02-Jul-1995': 7892436, '03-Jul-1995': 11739190, '04-Jul-1995': 24976177, '05-Jul-1995': 22468066, '06-Jul-1995': 20419373, '07-Jul-1995': 9566244, '08-Jul-1995': 5475250, '09-Jul-1995': 4312672, '10-Jul-1995': 13195178, '11-Jul-1995': 22694805, '12-Jul-1995': 17861622, '13-Jul-1995': 15959344, '14-Jul-1995': 16143956, '15-Jul-1995': 17900110, '16-Jul-1995': 8086988, '17-Jul-1995': 18423405, '18-Jul-1995': 17947142, '19-Jul-1995': 16164044, '20-Jul-1995': 25504026, '21-Jul-1995': 25910651, '22-Jul-1995': 6224677, '23-Jul-1995': 10089651, '24-Jul-1995': 20554069, '25-Jul-1995': 23269918, '26-Jul-1995': 26191814, '27-Jul-1995': 22953465, '28-Jul-1995': 37450120, '29-Jul-1995': 16285535, '30-Jul-1995': 21147546, '31-Jul-1995': 29820800}


C:\Users\VISHNU\AppData\Local\Temp\ipykernel_16656\1459748089.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_july['day'] = df_july['timestamp'].dt.strftime('%d-%b-%Y')


In [58]:
#8 Hourly request distribution

In [59]:
# Convert timestamp column to datetime format (invalid values become NaT)
final_df['timestamp'] = pd.to_datetime(final_df['timestamp'], errors='coerce')

# Extract the hour (0–23) from timestamp and store it in a new column 'hour'
final_df['hour'] = final_df['timestamp'].dt.hour

# Group data by hour and count number of requests per hour, then convert result to dictionary
result_dict = final_df.groupby('hour').size().to_dict()

# Print the dictionary showing hour-wise request count
print(result_dict)

{0: 39500, 1: 32525, 2: 30636, 3: 28083, 4: 25914, 5: 22744, 6: 19705, 7: 16916, 8: 13808, 9: 11381, 10: 10537, 11: 10366, 12: 12261, 13: 15145, 14: 22029, 15: 30859, 16: 37916, 17: 46220, 18: 45658, 19: 49936, 20: 50957, 21: 52780, 22: 50319, 23: 46086}


C:\Users\VISHNU\AppData\Local\Temp\ipykernel_16656\2805737664.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['timestamp'] = pd.to_datetime(final_df['timestamp'], errors='coerce')
C:\Users\VISHNU\AppData\Local\Temp\ipykernel_16656\2805737664.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['hour'] = final_df['timestamp'].dt.hour


In [60]:
#9 Top 10 most requested filenames

In [61]:
# Count frequency of each filename, get top 10 most accessed files, and convert to (filename, count) pairs
result = list(final_df['filename'].value_counts().head(10).items())

# Print the top 10 most frequently accessed filenames
print(result)

[('index.html', 139505), ('3.gif', 24006), ('2.gif', 23595), ('4.gif', 8018), ('244.gif', 5148), ('5.html', 5009), ('4097.gif', 4874), ('8870.jpg', 4492), ('6733.gif', 4278), ('8472.gif', 3843)]


In [62]:
#10 HTTP response code distribution

In [63]:
# Count occurrences of each HTTP status code and convert the result into a dictionary
result_dict = final_df['http_code'].value_counts().to_dict()

# Print the dictionary showing frequency of each HTTP status code
print(result_dict)

{200: 566139, 304: 97560, 302: 30258, 404: 23440, 403: 4740, 401: 46, 501: 43, 500: 42, 400: 13}
